In [1]:
import time
import os

import porespy as ps
import numpy as np
import scipy as sc
from pypardiso import spsolve
from scipy.sparse import csc_matrix, csr_matrix

os.chdir("..")
%run .\pyflowsolver\volumeManager.py
%run .\pyflowsolver\sparseArray.py
%run .\pyflowsolver\fastLaplacian.py
%run .\pyflowsolver\darcySolver.py
os.chdir("notebooks")

In [2]:
SIZE = 50
vol = ps.generators.blobs(shape=(SIZE, SIZE, SIZE), blobiness=0.4, porosity=0.55)
vol, n_lab = sc.ndimage.label(vol)
vol = (vol==1)
if vol.sum() == 0:
    print('error')
else:
    print('image OK')

cond_vol = (vol==1)*100 #porosity map ndarray uint8 0..100
cond_vol = fast_laplacian_volume_generator(
    cond_vol, 
    (1., 1., 1.), 
    )

#cond_vol[:cond_vol.shape[0]//2, :, :] *= 0.0000001
cond_vol[:cond_vol.shape[0]//2, :, :] *= 0.00001

volume_manager = VolumeManager(cond_vol)

image OK


In [3]:
if vol.shape[0] <= 30: # 30 for a 64 Gb RAM system
    dense_A, dense_b = volume_manager.get_linear_system()
    solution_template = np.linalg.solve(dense_A, dense_b)
    raveled_template = volume_manager.ravel_dense_solution(solution_template)
else:
    raveled_template = None

In [5]:
solver = DarcySolver()
sparse_A, sparse_b = volume_manager.get_sparse_system_jit()

In [5]:
if sparse_b.size < 150000:
    start_time = time.perf_counter()
    solution, error, iterations = solver.solve_jit(
        sparse_A, 
        sparse_b, 
        parallel=12, 
        max_iterations=100000, 
        target_error=1e-6,
    )
    run_time = time.perf_counter() - start_time
    if raveled_template is not None:
        raveled_solution = volume_manager.ravel_sparse_solution(solution)
        diff = np.abs(raveled_solution - raveled_template)
        print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
    else:
        print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

Error: [9.3239834e-07]   Iterations: 37   Mean Error: 0.004041873384267092   Max error: 0.013702154159545898   Run time: 3.96431119996123


In [6]:
start_time = time.perf_counter()
max_iterations = sparse_b.size
solution, error, iterations = solver._solve_cg(
        sparse_A.val,
        sparse_A.col_idx,
        sparse_A.row_ptr,
        sparse_b,
        max_iterations=max_iterations*100, # sqrt(n) for n x n system
        target_error=1.0e-9, # 1.0e-6
        X0=np.zeros(sparse_b.size, dtype=np.float64),
        threads=1,
    )
run_time = time.perf_counter() - start_time
if raveled_template is not None:
    raveled_solution = volume_manager.ravel_sparse_solution(solution)
    diff = np.abs(raveled_solution - raveled_template)
    print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
else:
    print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

Error: 6.808252525136924e-11   Iterations: 33   Mean Error: 2.2765662688328803e-09   Max error: 2.9802322387695312e-08   Run time: 3.8723855999996886


In [7]:
P_val, P_col_idx, P_row_ptr = _get_diagonal_preconditioner(
    A_val = sparse_A.val, 
    A_col_idx=sparse_A.col_idx, 
    A_row_ptr=sparse_A.row_ptr, 
    threads=1,
    )
print(P_val)
print(P_col_idx)
print(P_row_ptr)

start_time = time.perf_counter()
max_iterations = sparse_b.size
solution, error, iterations = solver._solve_pcg(
        sparse_A.val,
        sparse_A.col_idx,
        sparse_A.row_ptr,
        P_val, 
        P_col_idx, 
        P_row_ptr,
        sparse_b,
        max_iterations=max_iterations*100, # sqrt(n) for n x n system
        target_error=1.0e-9, # 1.0e-6
        X0=np.zeros(sparse_b.size, dtype=np.float64),
        threads=1,
    )
run_time = time.perf_counter() - start_time
if raveled_template is not None:
    raveled_solution = volume_manager.ravel_sparse_solution(solution)
    diff = np.abs(raveled_solution - raveled_template)
    print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
else:
    print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

[-1.00000003e+05 -1.00000503e+05 -8.00003222e+04 -6.66668907e+04
 -1.00000003e+05 -1.33334226e+05 -1.99998000e+00 -1.33332444e+00
 -1.33332444e+00 -1.00000000e+00 -1.99998000e+00 -1.00000000e+00
 -2.00000000e+00 -1.00000000e+00 -1.00000000e+00]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
Error: 5.56113660563371e-10   Iterations: 12   Mean Error: 1.910246112402092e-07   Max error: 1.8253922462463379e-06   Run time: 1.707417199970223


In [6]:
A = csr_matrix( (sparse_A.val, sparse_A.col_idx, np.append(sparse_A.row_ptr,sparse_A.val.size)) )

In [9]:
spsolve(A, sparse_b)

array([9.99941586e-01, 9.99820156e-01, 9.99717757e-01, ...,
       2.58266384e-04, 1.48430687e-04, 4.73645888e-05])

In [11]:
start_time = time.perf_counter()
max_iterations = sparse_b.size
A = csr_matrix( (sparse_A.val, sparse_A.col_idx, np.append(sparse_A.row_ptr,sparse_A.val.size)) )
solution = spsolve(A, sparse_b)
error = 0
iterations = 0
run_time = time.perf_counter() - start_time
if raveled_template is not None:
    raveled_solution = volume_manager.ravel_sparse_solution(solution)
    diff = np.abs(raveled_solution - raveled_template)
    print(f"Error: {error}   Iterations: {iterations}   Mean Error: {diff.mean()}   Max error: {diff.max()}   Run time: {run_time}")
else:
    print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")

Error: 0   Iterations: 0   Mean Error: 2.2765662688328803e-09   Max error: 2.9802322387695312e-08   Run time: 0.00044299999717622995


In [12]:
solution

array([0.79803257, 0.32820107, 0.09371093, 0.86392919, 0.81828689,
       0.40921838, 0.21052922, 0.07017656, 0.78362704, 0.35087952,
       0.20468208, 0.05848055, 0.20467975, 0.05847997, 0.02924013])

In [33]:
SIZE = 6
vol = np.zeros((SIZE,)*3, dtype=np.uint8)
vol[2:-2, 2:-2, :] = 1
vol[1:-1, 1:-1, 2:4] = 1

cond_vol = vol*100 #porosity map ndarray uint8 0..100
cond_vol = fast_laplacian_volume_generator(
    cond_vol, 
    (1., 1., 1.), 
    )

#cond_vol[:cond_vol.shape[0]//2, :, :] *= 0.0000001
#cond_vol[:cond_vol.shape[0]//2, :, :] *= 0.00001

volume_manager = VolumeManager(cond_vol)

solver = DarcySolver()
sparse_A, sparse_b = volume_manager.get_sparse_system_jit()

In [34]:
start_time = time.perf_counter()
max_iterations = sparse_b.size
A = csr_matrix( (sparse_A.val, sparse_A.col_idx, np.append(sparse_A.row_ptr,sparse_A.val.size)) )
solution = spsolve(A, sparse_b)
error = 0
iterations = 0
run_time = time.perf_counter() - start_time
print(f"Error: {error}   Iterations: {iterations}   Run time: {run_time}")
raveled_solution = volume_manager.ravel_sparse_solution(solution)

Error: 0   Iterations: 0   Run time: 0.0009687999845482409


In [35]:
raveled_solution

array([[[0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ]],

       [[0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.5048939 , 0.4951061 , 0.        ,
         0.        ],
        [0.        , 0.        , 0.5097878 , 0.49021217, 0.        ,
         0.        ],
        [0.        , 0.        , 0.5097878 , 0.49021217, 0.        ,
         0.        ],
        [0.        , 0.        , 0.5048939 , 0.4951061 , 0.        ,
         0.        

In [36]:
scale = (1., 1., 1.)
size = raveled_solution.shape
scaled_size = [(a*b) for (a,b) in zip(scale, size)]

x_gradient = np.zeros_like(raveled_solution)
x_gradient[:-1,:,:] = raveled_solution[:-1,:,:] - raveled_solution[1:,:,:]
x_gradient[-1,:,] = raveled_solution[-1,:,:]
x_cond = np.zeros_like(raveled_solution)
x_cond[:-1,:,:] = 2 / (1/cond_vol[:-1,:,:] + 1/cond_vol[1:,:,:])
x_cond[-1,:,:] = 0
x_speed = x_gradient * x_cond
q_x = (x_speed[0,:,:].sum() + x_speed[-1,:,:].sum())/2
k_x = (q_x * scaled_size[0]) / (scaled_size[1] * scaled_size[2]) # must be zero or close

y_gradient = np.zeros_like(raveled_solution)
y_gradient[:,:-1,:] = raveled_solution[:,:-1,:] - raveled_solution[:,1:,:]
y_gradient[:,-1,:] = raveled_solution[:,-1,:]
y_cond = np.zeros_like(raveled_solution)
y_cond[:,:-1,:] = 2 / (1/cond_vol[:,:-1,:] + 1/cond_vol[:,1:,:])
y_cond[:,-1,:] = 2 * cond_vol[:,-1,:]
y_speed = y_gradient * y_cond
q_y = (y_speed[:,0,:].sum() + y_speed[:,-1,:].sum())/2
k_y = (q_y * scaled_size[1]) / (scaled_size[0] * scaled_size[2])

z_gradient = np.zeros_like(raveled_solution)
z_gradient[:,:,:-1] = raveled_solution[:,:,:-1] - raveled_solution[:,:,1:]
z_gradient[:,:,-1] = raveled_solution[:,:,-1]
z_cond = np.zeros_like(raveled_solution)
z_cond[:,:,:-1] = 2 / (1/cond_vol[:,:,:-1] + 1/cond_vol[:,:,1:])
z_cond[:,:,-1] = 2 * cond_vol[:,:,-1]
z_speed = z_gradient * z_cond
q_z = (z_speed[:,:,0].sum() + z_speed[:,:,-1].sum())/2
k_z = (q_z * scaled_size[2]) / (scaled_size[0] * scaled_size[1])


In [37]:
print(k_x, k_y, k_z)

0.0 0.0 0.03677635143200556


In [38]:
z_speed[:,:,0].sum()

0.22065812

In [39]:
z_speed[:,:,-1].sum()

0.2206581

In [40]:
z_speed

array([[[ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ]],

       [[ 0.        ,  0.        ,  0.        ,  0.        ,
          0.        ,  0.        ],
        [ 0.        , -0.        ,  0.00244695,  0.        ,
          0.        ,  0.        ],
        [ 0.        , -0.        ,  0.00489391,  0.        ,
          0.        ,  0.        ],
        [ 0.        , -0.        ,  0.00489391,  0.        ,
          0.        ,  0.        ],
        [ 0.        , -0.   

In [41]:
z_cond

array([[[0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ]],

       [[0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.25      , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.25      , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.25      , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.25      , 0.        , 0.        ,
         0.        

In [42]:
raveled_solution

array([[[0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ]],

       [[0.        , 0.        , 0.        , 0.        , 0.        ,
         0.        ],
        [0.        , 0.        , 0.5048939 , 0.4951061 , 0.        ,
         0.        ],
        [0.        , 0.        , 0.5097878 , 0.49021217, 0.        ,
         0.        ],
        [0.        , 0.        , 0.5097878 , 0.49021217, 0.        ,
         0.        ],
        [0.        , 0.        , 0.5048939 , 0.4951061 , 0.        ,
         0.        

In [43]:
z_speed.sum(axis=(0,1))

array([0.22065812, 0.22065803, 0.2206581 , 0.22065812, 0.22065808,
       0.2206581 ], dtype=float32)